In [76]:
def pavage_1d(tailles, longueur):
    if longueur == 0: return []
    for i in range(len(tailles)):
        if longueur >= tailles[i] and tailles[i] > 0:
            if r is not None: return r+[tailles[i]]
    return None

In [77]:
pavage_1d([3, 2, 1], 20)

TypeError: unsupported operand type(s) for +: 'R' and 'list'

In [1]:
cols = sorted(colors)
cols.remove("beige")
cols.remove("antiquewhite")
cols.remove("white")
cols.remove("automatic")
cols.remove("blanchedalmond")
cols.remove("aqua")
i = 0
while i < len(cols):
    if cols[i][:5] == "light":
        cols.pop(i)
    else:
        i += 1

In [2]:
def draw_square(x, y, color="blue"):
    return polygon2d([(x, y), (x+1, y), (x+1, y+1), (x, y+1)], color=color)

def grille(width, height):
    return polygon2d([(0, 0), (width, 0), (width, height), (0, height)], color="red", fill=False)

def draw(pol, color="blue"):
    res = grille(pol.shape[1], pol.shape[0])
    y_max = pol.shape[0]
    pol = list(pol)
    for y in range(len(pol)):
        for x in range(len(pol[y])):
            if pol[y][x] == 1:
                res += draw_square(x, y_max-y-1, color=color)
    return res

def draw_with_colors(pol):
    res = grille(pol.shape[1], pol.shape[0])
    y_max = pol.shape[0]
    pol = list(pol)
    for y in range(len(pol)):
        for x in range(len(pol[y])):
            if pol[y][x] >= 1:
                res += draw_square(x, y_max-y-1, color=cols[pol[y][x]% len(cols)])
    return res

In [3]:
import numpy as np
def to_square(pol):
    res = np.zeros((max(pol.shape), max(pol.shape)), dtype="int64")
    res[:pol.shape[0], :pol.shape[1]] = pol
    return res

def generate(pol):
    x_min, x_max, y_min, y_max = -1, -1, -1, -1
    for x in range(pol.shape[1]):
        if pol[:,x].max() == 1 and x_min == -1:
            x_min = x
        if pol[:,-x-1].max() == 1 and x_max == -1:
            x_max = pol.shape[1] - x
        if x_max != -1 and x_min != -1: break
    for y in range(pol.shape[0]):
        if pol[y,:].max() == 1 and y_min == -1:
            y_min = y
        if pol[-y-1,:].max() == 1 and y_max == -1:
            y_max = pol.shape[0] - y
        if y_max != -1 and y_min != -1: break
    if x_min == -1 or y_min == -1: 
        return np.ones((0, 0))
    return to_square(pol[y_min:y_max, x_min:x_max])

In [6]:
"""
Ces fonctions ne marchent que si pol est de taille plus petite possible (pas forcément carré donc)
"""
def target(surface):
    liste_y = np.argmin(surface, axis=1)
    for y in reversed(range(len(liste_y))):
        x = liste_y[y]
        if surface[y, x] == 0:
            return x, y
    return None
        
def addable(surface, pol, pos):
    if pos[0] < 0 or pos[1] < 0: return False
    def func(x): return 1 if x > 0 else 0
    surface = np.vectorize(func)(surface)
    t = surface[pos[1]:pos[1]+pol.shape[0], pos[0]:pos[0]+pol.shape[1]]
    return pol.shape == t.shape and (pol + t).max() != 2

animation = []

def pavage_2d(surface, pols, nb=0):
    global current_col
    #print(surface, end="\n\n")
    animation.append(draw_with_colors(surface))

    t = target(surface)
    if t is None:
        return surface
    x, y = t
    for pol in pols:
        x_moins = np.argmax(pol[-1])
        pos1 = (x-x_moins, y-pol.shape[0]+1)
        pos2 = (pos1[0]+pol.shape[1], pos1[1]+pol.shape[0])
        if addable(surface, pol, pos1):
            new_surf = surface.copy()
            new_surf[pos1[1]:pos2[1], pos1[0]:pos2[0]] += pol*nb
            p = pavage_2d(new_surf, pols, nb+1)
            if p is not None: return p
    return None
        

"""
pavage_2d(
np.zeros((8, 8), dtype="int64"),
[np.array([[1, 1],[1, 0]]),
 np.array([[1, 0],[1, 0]]),
 np.array([[1, 1]])]
)
#animate(animation).show()
animate(animation).apng(delay=75, show_path=True)
#animate(animation).show(use_ffmpeg=True)
"""

'\npavage_2d(\nnp.zeros((8, 8), dtype="int64"),\n[np.array([[1, 1],[1, 0]]),\n np.array([[1, 0],[1, 0]]),\n np.array([[1, 1]])]\n)\n#animate(animation).show()\nanimate(animation).apng(delay=75, show_path=True)\n#animate(animation).show(use_ffmpeg=True)\n'

In [25]:
import numpy as np
def gravite(surface, polyomino, pos_x, col=1):
    for y in range(surface.shape[0]):
        if not addable(surface, polyomino, (pos_x, y)):
            surface[y-1:y+polyomino.shape[0]-1, pos_x:polyomino.shape[1]+pos_x] += polyomino * col
            return pos_x, y-1
    surface[surface.shape[0]-polyomino.shape[0]:surface.shape[0], pos_x:polyomino.shape[1]+pos_x] += polyomino * col
    return pos_x, surface.shape[0]-polyomino.shape[0]-1
            
def calcul_tangente(p1, p2):
    if p1[0] == p2[0]: return -1
    return -(p2[1] - p1[1]) / (p2[0] - p1[0])

def calcul_tangentes(pol):
    surf = np.zeros((pol.shape[0]*3, pol.shape[1]*3))
    gravite(surf, pol, 0)
    tang = []
    p1 = (0, surf.shape[0]-pol.shape[0])
    for x in range(pol.shape[1]+1):
        surface = np.copy(surf)
        p2 = gravite(surface, pol, x)
        tang.append(calcul_tangente(p1, p2))
    return tang
        
animation = []
    
def pont(pB, pol):
    surface = np.zeros((pB[1]+pol.shape[0]+1, pB[0]+pol.shape[1]+1), dtype=int)
    tangentes = calcul_tangentes(pol)
    prev_pos = gravite(surface, pol, 0)
    animation.append(draw_with_colors(surface))
    pB = pB[0]-1, 2
    col = 2
    while prev_pos[1] > pB[1]:
        tang = calcul_tangente(prev_pos, pB)
        print(prev_pos, pB)
        if tang == -1: prev_pos = gravite(surface, pol, prev_pos[0], col)
        else:
            max_i = None
            for i in range(1, len(tangentes)):
                if ((max_i is None or tangentes[i] >= tangentes[max_i]) and tangentes[i] <= tang):
                    max_i = i
            if max_i is None: max_i = len(tangentes)-1
            prev_pos = gravite(surface, pol, max_i+prev_pos[0], col)
        col += 1
        animation.append(draw_with_colors(surface))

In [83]:
a = np.zeros((10, 10))
gravite(a, np.array([[1,1,1,],[1,1,1],[0,1,0]]), 0)
gravite(a, np.array([[1,1,1,],[1,1,1],[0,1,0]]), 2)
print(a)

[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 1. 1. 0. 0. 0. 0. 0.]
 [0. 0. 1. 1. 1. 0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 0. 0. 0. 0. 0. 0.]
 [1. 1. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]]


In [32]:
animation = []
pont((100, 25), np.array([[1,1,1,1,1], [1,0,0,0,1], [1,0,0,0,1], [1,0,0,0,1]]))
animate(animation).apng(delay=10, show_path=True)

(0, 26) (99, 2)
(5, 26) (99, 2)
(10, 26) (99, 2)
(15, 26) (99, 2)
(20, 26) (99, 2)
(25, 26) (99, 2)
(30, 26) (99, 2)
(35, 26) (99, 2)
(40, 26) (99, 2)
(45, 26) (99, 2)
(50, 26) (99, 2)
(55, 26) (99, 2)
(60, 26) (99, 2)
(65, 26) (99, 2)
(70, 26) (99, 2)
(75, 26) (99, 2)
(79, 22) (99, 2)
(83, 18) (99, 2)
(87, 14) (99, 2)
(91, 10) (99, 2)
(95, 6) (99, 2)
Animation saved to file /tmp/tmppm3107ds/tmp_ldgukc7o.png.


In [15]:
print(np.array([[1]]).shape)

(1, 1)
